In [120]:
import pandas as pd
import glob
from scipy import signal
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch import nn,optim
from torch.nn import functional as F
from scipy import stats
from sklearn import preprocessing
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn import datasets, linear_model
from sklearn.model_selection import cross_val_score
from tqdm import tqdm
from sklearn.utils import shuffle
import math
from keras.utils import np_utils
from scipy.stats import norm,kurtosis,skew



In [121]:
###
# All variables releated data are located here to make an order
###
le=preprocessing.LabelEncoder()
LABELS=['grazing','running','standing','trotting','walking-natural','walking-rider']
PATH="C:\\Users\\emirc\\Desktop\\AAR\\data_/*" #Refer the path having the csv files
OUTPUT_LABEL='ActivityEncoded'

REMOVE_COLUMNS=['Mx','My','Mz','Gx','Gy','Gz','G3D','M3D','A3D','segment'] #Columns to delete such as Gx Gy Gz
TIME_PERIODS = 200
STEP_DISTANCE = 100
FILES = sorted(glob.glob(PATH))
N_FEATURES=3

In [122]:
def create_dataframe(files):
    result=pd.DataFrame()    
    for file in tqdm(files):
        csv=pd.read_csv(file)
        result=result.append(csv)
    
    result.drop(REMOVE_COLUMNS, axis=1, inplace=True)
    result=merge_activities(result)
    result[OUTPUT_LABEL]=le.fit_transform(result['label'].values.ravel())
    return result

In [123]:
def merge_activities(df):
    df['label']=df['label'].replace(to_replace=['trotting-natural'], value='trotting')
    df['label']=df['label'].replace(to_replace=['trotting-rider'], value='trotting')
    df['label']=df['label'].replace(to_replace=['running-natural'], value='running')
    df['label']=df['label'].replace(to_replace=['running-rider'], value='running')
    result=df[df['label'].isin(LABELS)]
    return result

In [124]:
def filter(df):
    sos = signal.butter(N=3, Wn=30, btype='lowpass', fs=100, output='sos')
    df['Ax'] = signal.sosfilt(sos, df['Ax'])
    df['Ay'] = signal.sosfilt(sos, df['Ay'])
    df['Az'] = signal.sosfilt(sos, df['Az'])
    return df
    

In [125]:
def split_by_subject(df, name):
    test=df[df['filename'].str.contains(name)]
    train=df[~df['filename'].str.contains(name)]
    return train, test

In [126]:
def feature_scaling(data):
    train_x_max = data['Ax'].max()
    train_y_max = data['Ay'].max()
    train_z_max = data['Az'].max()

    pd.options.mode.chained_assignment = None

    #divide all 3 axis with the max value in the training set
    data['Ax'] = data['Ax'] / train_x_max
    data['Ay'] = data['Ay'] / train_y_max
    data['Az'] = data['Az'] / train_z_max

    return data

In [127]:
def create_windows(df, time_steps, step, label_name):
    windows = []
    labels = []
    for i in range(0, len(df)-time_steps, step):
        axs = df['Ax'].values[i: i + time_steps]
        ays = df['Ay'].values[i: i + time_steps]
        azs = df['Az'].values[i: i + time_steps]
        # Retrieve the most often used label in this segment
        label = stats.mode(df[label_name][i: i + time_steps])[0][0]
        windows.append([axs, ays, azs])
        labels.append(label)
    # Bring the segments into a better shape
    reshaped_windows = np.asarray(windows, dtype= np.float32).reshape(-1, time_steps, N_FEATURES)
    labels = np.asarray(labels)

    return reshaped_windows, labels

In [128]:
def getStatisticalFeatures(data,name):
    columns=['mean'+name,'std'+name,'median'+name,'cov'+name,'twf_perc'+name,'svf_perc'+name,'maxi'+name,'mini'+name,'skewness'+name,'kurt'+name]
    res=pd.DataFrame(columns=columns)
    for i, window in tqdm(enumerate(data)):
        maxi=np.amax(window)
        mini=np.amin(window)
        mean=np.mean(window, axis=None)
        std=np.std(window, axis=None)
        median=np.median(window, axis=None)
        cov=math.sqrt(std)
        twf_perc=np.percentile(window, 25, axis=None)
        svf_perc=np.percentile(window, 75, axis=None)
        skewness=skew(window)
        kurt=kurtosis(window)
        res=res.append({'mean'+name:mean,
                    'std'+name:std,
                    'median'+name:median,
                    'cov'+name:cov,
                    'twf_perc'+name:twf_perc,
                    'svf_perc'+name:svf_perc,
                    'maxi'+name:maxi,
                    'mini'+name:mini,
                    'skewness'+name:skewness,
                    'kurt'+name:kurt},ignore_index=True)
        
    return res


In [129]:
def reshape_input(x,shape): #??????
    result=x.reshape(x.shape[0],shape)
    return result

In [130]:
def get_latent_representations(data, w=True): #!
    nr_of_windows=data.shape[0]
    data=data.reshape(nr_of_windows,0)
    tensor=torch.tensor(data)
    trained=[]
    for i, window in tqdm(enumerate(tensor)):
        reps=MODEL(torch.unsqueeze(window, 0))
        reps=torch.detach(reps[1])
        trained.append(reps.numpy())
        trained[i]=np.full((200,3), reps)
    trained=np.asarray(trained)
    return trained
    

In [131]:
def encode_output(y, classes):
    result=np_utils.to_categorical(y,classes)
    return result

In [132]:
def preprocess_training(df, input_shape,num_classes):
    train=feature_scaling(df)
    x_train, y_train=create_windows(train, TIME_PERIODS,STEP_DISTANCE,OUTPUT_LABEL)
    x_train, y_train=shuffle(np.array(x_train),np.array(y_train))
    
    
    x_train=reshape_input(x_train,input_shape)
    x_train=x_train.astype('float32')
    y_train=y_train.astype('float32')
    y_train=encode_output(y_train,num_classes)
    return x_train, y_train


In [133]:
def preprocess_test(df, input_shape, num_classes):
    test=feature_scaling(df)
    x_test,y_test=create_windows(test, TIME_PERIODS,STEP_DISTANCE,OUTPUT_LABEL)
    x_test=get_latent_representations(x_test)
    x_test=x_test.astype('float32')
    y_test=y_test.astype('float32')
    y_test=encode_output(y_test,num_classes)
    return x_test, y_test

In [134]:
def preprocess(df,test_subject):
    input_shape=(TIME_PERIODS*N_FEATURES)
    num_classes=len(LABELS)
    df=filter(df)
    train, test=split_by_subject(df, test_subject)
    x_train, y_train=preprocess_training(train, input_shape, num_classes)
    #x_test, y_test=preprocess_test(test, input_shape,num_classes)
    return x_train, y_train,input_shape,num_classes


In [135]:
def getAllFeatures(df):
    featuresOfChosenLabels=[]
    for names in LABELS:
        featuresOfChosenLabels.append(featureExtraction(df,names))
    return featuresOfChosenLabels

In [136]:
def split_by_activity(df, activity):
    try:
        return df.loc[df['label']==activity]
    except KeyError:
        return

In [137]:
def featureExtraction(df, activity):
    df=split_by_activity(df,activity)
    train=feature_scaling(df)
    x_train, y_train=create_windows(train, TIME_PERIODS,STEP_DISTANCE,OUTPUT_LABEL)
    x_train, y_train=shuffle(np.array(x_train),np.array(y_train))
    #x_train=get_latent_representations(x_train)
    statistics=[]
    statistics.append(getStatisticalFeatures(x_train[::1],'_Ax'))
    statistics.append(getStatisticalFeatures(x_train[::2],'_AY'))
    statistics.append(getStatisticalFeatures(x_train[::3],'_Az'))
    result=statistics[0]
    result=statistics[0].merge(statistics[1],right_index=True,left_index=True)
    result=result.merge(statistics[2],right_index=True,left_index=True)
    return result

In [138]:
def createCleanCSV():
    df=create_dataframe(FILES)
    df.to_csv('dataset.csv')
    return df

In [139]:
createCleanCSV() #to locally store cleaned csv file for effectivity 

  0%|                                                                                           | 0/55 [00:00<?, ?it/s]C:\Users\emirc\AppData\Local\Temp/ipykernel_13624/2913561215.py:2: DtypeWarning: Columns (12) have mixed types.Specify dtype option on import or set low_memory=False.
  df=create_dataframe(FILES)
100%|██████████████████████████████████████████████████████████████████████████████████| 55/55 [03:19<00:00,  3.62s/it]


,Ax,Ay,Az,label,subject,ActivityEncoded
265689,6.474152,-1.182778,3.332848,walking-natural,11,3
265690,6.603444,-0.478857,3.227499,walking-natural,11,3
265691,6.656118,0.814058,3.524391,walking-natural,11,3
265692,6.842873,2.083030,3.821282,walking-natural,11,3
265693,6.828507,3.174825,4.180425,walking-natural,11,3
...,...,...,...,...,...,...
749840,13.000979,4.218734,-1.915430,walking-rider,8,4
749841,13.489414,3.931419,-1.144469,walking-rider,8,4
749842,13.360122,3.452562,-1.149258,walking-rider,8,4
749843,12.957882,3.031167,-0.588995,walking-rider,8,4


In [142]:
def getCleanCSV():
    df=pd.DataFrame()
    df=df.append(pd.read_csv('dataset.csv'))
    return df

In [146]:
df=getCleanCSV()
df=df.drop(df.columns[0], axis=1, inplace=True)


In [147]:
print(df)

None
